#### Step 1: Import Libraries & API Keys

In [1]:
import os
import groq
import json
import requests
import gradio as gr
from openai import OpenAI
from pprint import pprint
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")


c:\Users\Orchid X\OneDrive\Desktop\AI_Engineering\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Step 2: Set up Pushover

In [2]:
# a -> Set up account in your browser
# b -> Set up the app on your iPhone/Android phone, log into the same account
# c -> In the browser create an "Application/API token"
# d -> Copy your user key and API token into the .env file
# Like this but without the hashtag symbols and with your own keys:
# PUSHOVER_USER=xxxxxxxxxx
# PUSHOVER_TOKEN=yyyyyyyyy

# Save changes in the .env file

In [2]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [3]:

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)
    

In [ ]:
# send_notification("Hello to myself, from this amazing AI Engineering training.")

#### Step 3: Describe Pushover as an LLM tool

In [4]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. Use this to alert the user about important events, completed tasks, or time-sensitive information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message":{
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

#### Step 4: Add Pushover to the list of tools for the LLM

In [6]:
tools = [{"type": "function", "function": send_notification_function}]

#### Steps 2b, 3b and 4b: Create new function, describe it, add it to the list of tools

In [7]:
import random

# simulates rolling a single six-sided dice
def dice_roll(): 
    result = random.randint(1,6)
    return result

# Describe function function for the LLM
roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a six-sided die and returns the result. Use this when the user wants to roll a die for games, decisions, or random number generation.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

# Add function to list of tools of LLM
tools.append({"type": "function", "function": roll_dice_function})

#### Step 5: Calling the tool from an LLM

In [ ]:
def handle_tool_call(tool_calls):
    tool_results = []
    
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        
        
        # Route to the approproate function based on function_name
        if function_name == "send_notification":
            
        # Actually send the notification, i.e call the tool 
            send_notification(args["message"])
            content = f"Notification sent: {args["message"]}"
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
        # elif function_name == "insert_function_name_2":
        #     content = insert_function_name_2(args["message"])
        # elif function_name == "insert_function_name_3":
        #     content = insert_function_name_3(args["message"])
        # ....
        else:
            content = f"Unknown function: {function_name}"
        
        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
        
        tool_results.append(tool_call_result)
    return tool_results
    # return what to add to our "context" (about tool call results), a dictionary


In [10]:
client = OpenAI()
messages=[
        {
            "role":"user", "content": "Please do 2 things:\
                1) I'd like to roll two dice, and\
                2) Send me a notification with the highest of the rolls."
        }
    ]
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

message = response.choices[0].message
# print(message)
from pprint import pprint
# Check if model wants to call a tool
while message.tool_calls:
    pprint(message.tool_calls)
    tool_result = handle_tool_call(message.tool_calls) # whole list of tool calls on purpose
    messages.append(message) # .... add messages to "context", i.e, messages
    messages.extend(tool_result) # .... add info about tool call response to "context", i.e, messages. Changed from append to extend when we switched to multiple tool call handling
    
    # .... invoke the LLM one more time to get it's updated response
    response = client.chat.completions.create(
        
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )
    message = response.choices[0].message
    # print(message.content) # from  the new LLM response
    
print(message.content)

[ChatCompletionMessageFunctionToolCall(id='call_8kg10xViBHOCKCPUTTquqpAb', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_RWu1rnafSvgZasSdUzwkq2wD', function=Function(arguments='{}', name='dice_roll'), type='function')]
{}
{}
{}
{}
[ChatCompletionMessageFunctionToolCall(id='call_R7D0FVyH6Ec7K7lFFBnam4Zl', function=Function(arguments='{"message":"The highest roll of the two dice is 4."}', name='send_notification'), type='function')]
{'message': 'The highest roll of the two dice is 4.'}
{"message":"The highest roll of the two dice is 4."}
I rolled two dice and got 3 and 4. I have sent you a notification with the highest roll, which is 4.
